# End-to-End Classification Workshop

Today we put everything together. Part 1 introduces SVMs using simple 2D data where you can *see* how different classifiers draw boundaries. Part 2 walks through the complete ML workflow on a real dataset - the same workflow you'll use for your project.

Changes after Day 20 lecture (2026-03-24):

- [x] Switch 3-class Iris plot from petal to sepal features for better overlap between all three species
- [x] Add ROC AUC to dummy baseline alongside accuracy, with explanation of why constant predictor always scores 0.5
- [x] Rewrite duration/leakage section: distinguish feature availability from true data leakage, add prediction vs inference framing, add hypothetical target leakage example
- [x] Expand dataset intro with "Available before the call?" column, clarify prior vs current campaign features
- [x] Expand calibration section: add interpretation guidance, add CalibratedClassifierCV before/after demo for SVM
- [x] Expand "Is it surprising that logistic regression wins?" answer with explicit No Free Lunch connection

## Setup

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_iris, make_moons
from sklearn.dummy import DummyClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    make_scorer,
    RocCurveDisplay,
)
from sklearn.model_selection import (
    cross_val_score,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
    TunedThresholdClassifierCV,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
)
from sklearn.svm import SVC
from scipy.stats import loguniform

---

## Part 1: SVM and Decision Boundaries

We'll use the Iris dataset - the classic example for classification - with just two features so we can visualize how each classifier separates the classes. We start with two species that overlap (versicolor and virginica), then bring in all three for the kernel trick.

This section introduces the key ideas behind SVM. I don't expect deep intuition from a single lecture - the goal is familiarity with the concepts and the sklearn API. The theory is presented here for your reference; if you use classification in your work or research, SVM is worth understanding in depth.

### The Data

In [ ]:
iris = load_iris()

# Binary subset: versicolor (1) vs virginica (2), petal features only
# These two species overlap — unlike setosa, which is completely separated
mask = iris.target != 0
X_iris = iris.data[mask][:, 2:4]  # petal length, petal width
y_iris = iris.target[mask]

fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap="coolwarm", s=30, alpha=0.7)
ax.set_xlabel("Petal Length (cm)")
ax.set_ylabel("Petal Width (cm)")
ax.set_title("Iris: Versicolor vs. Virginica (overlapping classes)")
plt.tight_layout()
plt.show()

### Two Familiar Classifiers

Logistic regression draws a straight line that maximizes the likelihood of the labels. KNN draws a flexible boundary that follows local structure. Both are familiar from earlier in the course.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
familiar = {
    "Logistic Regression": LogisticRegression(),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
}
for ax, (name, model) in zip(axes, familiar.items()):
    model.fit(X_iris, y_iris)
    DecisionBoundaryDisplay.from_estimator(
        model, X_iris, ax=ax, cmap="coolwarm", alpha=0.3, response_method="predict"
    )
    ax.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap="coolwarm", s=30, edgecolors="k", linewidths=0.5)
    ax.set_xlabel("Petal Length (cm)")
    ax.set_ylabel("Petal Width (cm)")
    ax.set_title(name)
plt.suptitle("Decision Boundaries — Iris (Versicolor vs. Virginica)", y=1.02)
plt.tight_layout()
plt.show()

### How SVM Thinks

Logistic regression and SVM both draw linear boundaries, but they optimize different objectives.

Logistic regression maximizes the *likelihood* of the labels - it finds the boundary where the predicted probabilities best match the observed classes. Every data point contributes to the solution.

SVM takes a geometric approach. It finds the *widest street* (margin) that separates the two classes. The optimization problem is: maximize the margin width while penalizing points that fall on the wrong side. Formally, SVM minimizes:

$$\frac{1}{2} \|w\|^2 + C \sum_i \max(0, 1 - y_i(w \cdot x_i + b))$$

The first term keeps the margin wide (small $\|w\|$ = wide margin). The second is the *hinge loss* - it penalizes points that are either misclassified or too close to the boundary. Together, they balance margin width against classification accuracy.

The points that sit on or within the margin edges are called **support vectors**. The name comes from linear algebra: each data point is a vector in feature space, and these particular vectors "support" (define) the margin boundary. They're the only points that matter. Remove every other point and the boundary doesn't change. This *sparsity* property is unique to SVM.

In [ ]:
svm_linear = SVC(kernel="linear")
svm_linear.fit(X_iris, y_iris)

fig, ax = plt.subplots(figsize=(6, 5))
DecisionBoundaryDisplay.from_estimator(
    svm_linear, X_iris, ax=ax, cmap="coolwarm", alpha=0.3, response_method="predict"
)
ax.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap="coolwarm", s=30, edgecolors="k", linewidths=0.5)
# highlight support vectors
ax.scatter(
    svm_linear.support_vectors_[:, 0], svm_linear.support_vectors_[:, 1],
    s=120, facecolors="none", edgecolors="black", linewidths=2
)

# draw margin lines: decision_function = -1, 0, +1
xlim = ax.get_xlim()
ylim = ax.get_ylim()
xx = np.linspace(xlim[0], xlim[1], 200)
yy = np.linspace(ylim[0], ylim[1], 200)
XX, YY = np.meshgrid(xx, yy)
Z = svm_linear.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
ax.contour(XX, YY, Z, levels=[-1, 0, 1], linestyles=["--", "-", "--"], colors="k", linewidths=[1, 2, 1])

ax.set_xlabel("Petal Length (cm)")
ax.set_ylabel("Petal Width (cm)")
ax.set_title(f"SVM (linear) — {len(svm_linear.support_vectors_)} support vectors circled\n"
             f"Solid line: decision boundary. Dashed lines: margin edges.")
plt.tight_layout()
plt.show()

The circled points are the support vectors - the points on or within the margin. The solid line is the decision boundary (decision_function = 0), and the dashed lines show the margin edges (decision_function = -1 and +1). The "widest street" metaphor: the dashed lines are the curbs, the solid line is the center of the street, and the support vectors are the buildings that constrain how wide the street can be.

### The C Parameter

The **C parameter** controls the tradeoff in the optimization above. Small C allows a wide margin with some misclassifications (high bias, low variance). Large C enforces a narrow margin that tries to classify everything correctly (low bias, high variance). This is the same bias-variance tradeoff we've seen throughout the course.

This is analogous to k in KNN. There, k directly controls how many neighbors influence each prediction. Here, C indirectly controls how many support vectors define the boundary. Both determine how many data points shape the model's decisions - but they move in opposite directions: increasing k simplifies (more voters, smoother boundary), while increasing C complexifies (fewer support vectors, tighter boundary).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, C in zip(axes, [0.01, 1, 100]):
    svm = SVC(kernel="linear", C=C)
    svm.fit(X_iris, y_iris)
    DecisionBoundaryDisplay.from_estimator(
        svm, X_iris, ax=ax, cmap="coolwarm", alpha=0.3, response_method="predict"
    )
    ax.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap="coolwarm", s=30, edgecolors="k", linewidths=0.5)
    ax.scatter(
        svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
        s=100, facecolors="none", edgecolors="black", linewidths=1.5
    )
    # draw margin lines
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xx = np.linspace(xlim[0], xlim[1], 200)
    yy = np.linspace(ylim[0], ylim[1], 200)
    XX, YY = np.meshgrid(xx, yy)
    Z = svm.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    ax.contour(XX, YY, Z, levels=[-1, 0, 1], linestyles=["--", "-", "--"], colors="k", linewidths=[0.5, 1.5, 0.5])
    ax.set_xlabel("Petal Length (cm)")
    ax.set_ylabel("Petal Width (cm)")
    ax.set_title(f"C={C} ({len(svm.support_vectors_)} support vectors)")
plt.suptitle("C Parameter: Margin Width vs. Misclassification Penalty", y=1.02)
plt.tight_layout()
plt.show()

##### What happens to the number of support vectors as C increases? Why?

Small C prioritizes a wide margin, so many points fall within the margin and count as support vectors. Large C penalizes margin violations heavily, shrinking the margin so fewer points end up on the boundary. In the extreme (very large C), only the points closest to the opposing class define the margin.

### The Kernel Trick

Linear boundaries handle versicolor vs. virginica reasonably well with petal features, but what happens when we switch to sepal measurements and bring in all three species? Sepal length and width give much more overlap between all three classes - setosa is no longer trivially separated, and the three-way confusion makes linear boundaries visibly inadequate.

In [ ]:
# All three Iris classes, sepal features (more overlap than petals)
X_iris3 = iris.data[:, 0:2]  # sepal length, sepal width
y_iris3 = iris.target

fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(X_iris3[:, 0], X_iris3[:, 1], c=y_iris3, cmap="viridis", s=30, alpha=0.7)
ax.set_xlabel("Sepal Length (cm)")
ax.set_ylabel("Sepal Width (cm)")
ax.set_title("Iris: All Three Species (sepal features)")
handles, _ = scatter.legend_elements()
ax.legend(handles=handles, labels=["Setosa", "Versicolor", "Virginica"], title="Species")
plt.tight_layout()
plt.show()

The **kernel trick** is the key idea that makes SVM powerful beyond linear problems. The intuition: if data isn't separable in the original feature space, map it to a *higher-dimensional* space where it might be. For example, two clusters on a line (1D) that overlap might separate cleanly if you add a second dimension based on distance from the center.

The mathematical insight is that SVM's optimization only needs the *dot products* between data points, not the actual coordinates. A kernel function computes these dot products in the higher-dimensional space *without actually transforming the data*. This makes the computation tractable even when the implied feature space is infinite-dimensional.

The `rbf` (radial basis function) kernel is the most common: $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$. It measures similarity based on distance - points nearby in feature space get similar treatment, effectively creating flexible, localized boundaries.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_3class = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (rbf kernel)": SVC(kernel="rbf", C=10),
}
for ax, (name, model) in zip(axes, models_3class.items()):
    model.fit(X_iris3, y_iris3)
    DecisionBoundaryDisplay.from_estimator(
        model, X_iris3, ax=ax, cmap="viridis", alpha=0.3, response_method="predict"
    )
    ax.scatter(X_iris3[:, 0], X_iris3[:, 1], c=y_iris3, cmap="viridis", s=20, edgecolors="k", linewidths=0.5)
    ax.set_xlabel("Sepal Length (cm)")
    ax.set_ylabel("Sepal Width (cm)")
    ax.set_title(name)
plt.suptitle("Decision Boundaries — Iris (3 classes, sepal features)", y=1.02)
plt.tight_layout()
plt.show()

With sepal features, the three species overlap significantly and linear boundaries struggle across all three regions. KNN adapts to local structure but can be noisy. SVM with an rbf kernel curves its boundaries around the overlap zones, finding a smoother separation that captures the non-linear structure in the data.

This is also a callback to 09b: three classes means SVM uses the one-vs-one strategy internally - fitting a binary SVM for each pair of classes and combining the results.

##### Why does logistic regression draw straight boundaries while SVM(rbf) curves?

Logistic regression can only draw hyperplanes (straight lines in 2D). SVM with the rbf kernel implicitly maps the data to a higher-dimensional space where a linear boundary *can* separate the classes, then projects that boundary back. The result looks curved in the original feature space. KNN also handles non-linearity, but by memorizing local neighborhoods rather than finding a global boundary.

### SVM in Practice

For this course, the key takeaways:

- `SVC(kernel='linear')` for linear boundaries, `SVC(kernel='rbf')` for non-linear
- The C parameter controls complexity for both (bias-variance tradeoff)
- Other kernels exist (polynomial, sigmoid) and the kernel idea appears across machine learning, not just in SVMs
- SVM's main practical limitation is *speed* - training scales roughly $O(n^2)$ to $O(n^3)$, which is why we saw it struggle on MNIST's 60k samples in 09b

For a thorough treatment of SVM theory and implementation, see Appendix C of *Hands-On Machine Learning* (Géron), available as a [free online appendix](https://ageron.github.io/). It uses the Iris dataset as its running example.

### Bonus: Non-Linear Power

As a final demonstration, here's a synthetic dataset where the classes are interleaved half-moons - no linear boundary can separate them. This is where SVM with the rbf kernel really shines.

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, {
    "Logistic Regression": LogisticRegression(),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (rbf)": SVC(kernel="rbf", C=10),
}.items()):
    model.fit(X_moons, y_moons)
    DecisionBoundaryDisplay.from_estimator(
        model, X_moons, ax=ax, cmap="coolwarm", alpha=0.3, response_method="predict"
    )
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="coolwarm", s=20, edgecolors="k", linewidths=0.5)
    ax.set_title(name)
plt.suptitle("Decision Boundaries — Interleaved Half-Moons", y=1.02)
plt.tight_layout()
plt.show()

Logistic regression is stuck with a straight line. SVM(rbf) finds a smooth, curved boundary that follows the structure of the data. This is the kernel trick in action - the same algorithm, but operating in a space where the data *is* linearly separable.

---

## Part 2: End-to-End Workflow

Now we apply the complete ML workflow to a real dataset. This is the process you'll follow for your project.

### The Dataset: Bank Marketing

This dataset comes from direct marketing campaigns (phone calls) of a Portuguese banking institution. Our goal is to predict which clients are likely to subscribe, so the bank can decide who to call.

The training data contains completed campaign records, including what happened during and after some prior calls. Understanding what each feature represents - and *when* it becomes available - is critical for building a model that can actually be deployed. Specifically, any feature that won't be available when predicting the outcome of new calls cannot be used to train the model. This is where domain knowledge plays a key role in building valid and reliable models.

| Feature | Description | Available before the call? |
|---------|-------------|:-:|
| `age` | Client age | Yes |
| `job` | Type of job (admin, technician, entrepreneur, ...) | Yes |
| `marital` | Marital status | Yes |
| `education` | Education level | Yes |
| `default` | Has credit in default? | Yes |
| `housing` | Has housing loan? | Yes |
| `loan` | Has personal loan? | Yes |
| `contact` | Contact communication type (cellular, telephone) | Yes |
| `month`, `day_of_week` | Last contact timing | Yes |
| `duration` | Last contact duration (seconds) | No |
| `campaign` | Number of contacts during this campaign | During |
| `pdays` | Days since last contact from a *previous* campaign (999 = never) | Yes |
| `previous` | Number of contacts in *previous* campaigns | Yes |
| `poutcome` | Outcome of the *previous* campaign | Yes |
| `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed` | Macroeconomic indicators | Yes |
| **`y`** | **Did the client subscribe? (yes/no)** | |

Note that `pdays`, `previous`, and `poutcome` refer to *previous* campaigns. An observation with `pdays = 999` represents first contact with that customer. About 96% of observations are first contact, so `previous` and `poutcome` are mostly empty. The predictive power in this dataset comes largely from demographics and macroeconomic indicators, not call-related behavioral features.

Source: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing) (Moro et al., 2014).

### The Workflow

Every supervised ML project follows the same structure:

1. **Hold out test data** - split once at the start, don't touch the test set until the very end
2. **Explore and engineer features** - using training data only
3. **Select a model via cross-validation** - the training data plays both roles (train folds and validation folds); the outcome is a *best model*: the model family and hyperparameters
4. **Refit on all training data** - the best model form + hyperparameters, now fit on all available training data; the outcome is the *best fit*: learned parameters from maximum data
5. **Final evaluation on held-out test** - one shot; this is your estimate of real-world performance

```text
All Data
├── Test (hold out — don't touch until step 5)
└── Train
    ├── CV Fold 1: train / val  ─┐
    ├── CV Fold 2: train / val   ├─→ Best Model (family + hyperparameters)
    ├── ...                      │
    └── CV Fold k: train / val  ─┘
                                  ↓
    Refit on ALL Train ──────────→ Best Fit (family + hyperparams + learned params)
                                  ↓
    Evaluate on Test ────────────→ Performance Estimate
```

Inside cross-validation, the training data is temporarily split into train folds and validation folds. Each fold takes a turn as the validation set, and the scores are averaged. This is why some references describe a three-way split (train / validation / test) - the validation set serves the same purpose as our CV folds. Cross-validation makes a separate validation set unnecessary by rotating that role through the training data.

### Step 1: Hold Out Test Data

In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/selva86/datasets/master/bank-full.csv",
    sep=";"
)
print(f"Full dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target distribution:\n{df['y'].value_counts()}")
print(f"\nMinority class: {(df['y'] == 'yes').mean():.1%}")

In [ ]:
# Subsample for manageable SVM training time
df = df.sample(n=5000, random_state=42).reset_index(drop=True)
print(f"Working dataset: {df.shape[0]:,} rows")

In [ ]:
# Separate features and target
X = df.drop(columns="y")
y = (df["y"] == "yes").astype(int)

# Stratified split - preserves class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train):,} rows")
print(f"Test:  {len(X_test):,} rows (set aside — we won't touch this until the end)")
print(f"\nTrain class balance: {y_train.mean():.1%} positive")
print(f"Test class balance:  {y_test.mean():.1%} positive")

### Step 2: Explore and Engineer Features (Training Data Only)

Everything we learn about the data comes from the training set.

In [ ]:
X_train.head()

In [ ]:
X_train.describe().round(1)

In [ ]:
# Feature types
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print(f"Numeric ({len(numeric_cols)}):     {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")

#### Leakage Check: Drop `duration`

The `duration` column records how long the phone call lasted. It's highly correlated with the outcome - subscribers talk 2-3x longer - so it seems like an excellent predictor. But `duration` cannot be recorded until after a call is completed, so it is of no use for prediction. If you ultimately want to decide who to call based on the likelihood of making a sale, the `duration` of the call cannot be known.

In [ ]:
print("Mean duration by outcome (training data):")
print(pd.DataFrame({
    "duration": X_train["duration"],
    "subscribed": y_train
}).groupby("subscribed")["duration"].mean().round(0))
print("\nSubscribers talk 2-3x longer — real signal, but unavailable at prediction time.")

The dataset documentation calls this leakage, and many tutorials follow that convention. On closer inspection, it's a related but different and entirely practical problem. When planning who to call you don't know how long the call will last, so it can't be a predictor.

If the goal were inference instead of prediction, i.e., "what factors drove subscriptions in past campaigns," duration would be a legitimate feature. Longer calls correlating with subscriptions is a real finding that could inform how agents conduct calls. The distinction between prediction (features must exist at decision time) and inference (features just need to exist in the historical record) determines which columns are fair game.

True *data leakage* is a methodological error where information contaminates the training process - like fitting a scaler on the full dataset before splitting, or a feature that was determined with knowledge of the outcome.

Imagine this dataset included a `follow_up_scheduled` column indicating whether the agent scheduled a callback. Unlike `duration`, this feature *would* have a value at prediction time (you could look it up from the last campaign). But the agent scheduled that follow-up *because* the previous call went well, encoding their judgment about the outcome into `follow_up_scheduled`. That is a more clear cut example of target leakage.

The practical test for any suspicious feature has two parts: (1) does this column have a value at decision time? and (2) was that value determined independently of the outcome? `duration` fails the first check. A hypothetical `follow_up_scheduled` flag would pass the first but fail the second.

In [ ]:
X_train = X_train.drop(columns="duration")
X_test = X_test.drop(columns="duration")
numeric_cols.remove("duration")

#### Outliers

Look at the `describe()` output above. The `campaign` column (number of contacts during this campaign) has a median of 2 and a 75th percentile of 3, but a maximum of 56. That's extreme right skew.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X_train["campaign"], bins=50, edgecolor="black")
axes[0].set_xlabel("campaign (number of contacts)")
axes[0].set_ylabel("count")
axes[0].set_title("campaign — raw distribution")

axes[1].hist(X_train["campaign"].clip(upper=20), bins=20, edgecolor="black")
axes[1].set_xlabel("campaign (clipped at 20)")
axes[1].set_ylabel("count")
axes[1].set_title("campaign — clipped")

plt.tight_layout()
plt.show()

print(f"campaign > 10: {(X_train['campaign'] > 10).sum()} rows "
      f"({(X_train['campaign'] > 10).mean():.1%})")

Are these outliers noise or signal? Someone contacted 56 times is probably *not* going to subscribe - the bank kept calling and they kept saying no. That's real information, not an error. But the extreme values can distort distance-based models (KNN, SVM).

In regression, outliers were a nuisance - they distort OLS, inflate residuals, and motivate robust methods. In classification, the question is different: does this extreme value carry predictive signal, or is it just noise that will mislead the model?

Here, the extreme `campaign` values are real and informative (persistent non-subscribers), but their exact magnitude matters less than the pattern. We'll let `StandardScaler` handle this - it won't eliminate the skew, but scaling ensures these values don't dominate distance calculations.

If we had decided these values were noise, we'd have two options:

- *Drop the rows entirely*: `X_train = X_train[X_train["campaign"] <= 10]` - loses data, but removes distortion. Only makes sense when the extreme rows are errors or truly irrelevant.
- *Clip the values*: `X_train["campaign"] = X_train["campaign"].clip(upper=10)` - preserves all rows, caps the magnitude. Better when the extreme values are real but their exact size doesn't matter.

Neither is necessary here - we'll let the scaler handle it.

#### Handle `pdays` Sentinel Value

The `pdays` column records days since the client was last contacted. A value of 999 means "never contacted" - it's a sentinel, not a real number. Treating 999 as a numeric value would distort any distance-based or scaled model.

We'll use `FunctionTransformer` to convert this into a binary flag: was the client previously contacted or not?

In [ ]:
print(f"pdays == 999: {(X_train['pdays'] == 999).sum():,} / {len(X_train):,} "
      f"({(X_train['pdays'] == 999).mean():.1%})")

The following function works on a copy of the original data and returns the modified data. It is designed to be used in a pipeline. Specifically, it replaces the `pdays` column with `was_contacted`, a binary flag derived from the original. Nothing else is changed.

In [ ]:
# We'll encode this inside the pipeline later, but here's the logic:
# pdays → 1 if previously contacted (pdays != 999), else 0
def pdays_to_contacted(X):
    """Convert pdays sentinel (999 = not contacted) to binary flag."""
    result = X.copy()
    result["was_contacted"] = (result["pdays"] != 999).astype(int)
    result = result.drop(columns="pdays")
    return result

#### Handle `default` Column

The `default` column (has credit in default?) is 20.9% "unknown" with only 3 "yes" values in the full dataset. A feature that's mostly "no" and "unknown" with almost no "yes" carries little predictive signal.

This is a judgment call that depends on domain knowledge. We can't know from the data alone whether "unknown" is informative or just noise. In principle, we could test both options - the pipeline makes it easy to swap features in and out. But be careful with that flexibility: running too many experiments increases the risk of overfitting to noise, finding a result that looks good by chance rather than signal.

Here, the evidence points toward dropping it.

In [ ]:
print(X_train["default"].value_counts())

In [ ]:
X_train = X_train.drop(columns="default")
X_test = X_test.drop(columns="default")
categorical_cols.remove("default")

#### Missing Data: "Unknown" Values

Several categorical columns contain "unknown" entries. These aren't NaN, but they represent missing information.

In [ ]:
print("'unknown' counts in training data:")
for col in categorical_cols:
    n = (X_train[col] == "unknown").sum()
    if n > 0:
        print(f"  {col}: {n} ({n / len(X_train):.1%})")

In [ ]:
# Show a row with missing data before replacement
sample_idx = X_train[X_train["education"] == "unknown"].index[0]
print(f"Row {sample_idx} before imputation:")
print(X_train.loc[sample_idx, ["job", "marital", "education", "housing", "loan"]])

We have two reasonable options:

1. *Treat "unknown" as its own category* - let `OneHotEncoder` create an "unknown" indicator column. Simple, and defensible if the fact that data is missing is itself informative (e.g., someone who won't share their education level might behave differently).

2. *Impute the missing values* - replace "unknown" with the most common category using `SimpleImputer`. This assumes the data is missing at random and the "unknown" label carries no signal.

We'll use option 2 here to demonstrate `SimpleImputer` in a Pipeline. First, replace the "unknown" strings with `NaN` so that sklearn's imputation recognizes them as missing.

In [ ]:
# Replace "unknown" with NaN so SimpleImputer can handle them
X_train = X_train.replace("unknown", np.nan)
X_test = X_test.replace("unknown", np.nan)

print("NaN counts after replacement:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

#### Build the Preprocessing Pipeline

The pipeline has three layers, built from the inside out:

1. **`FunctionTransformer`** runs first on the raw data. It converts the `pdays` column (with its 999 sentinel) into a binary `was_contacted` flag and drops the original. After this step, `pdays` no longer exists in the data - which is why our numeric column list excludes it.

2. **`ColumnTransformer`** routes the *transformed* columns to their respective branches:
   - *Numeric branch*: `StandardScaler` - centers and scales continuous features
   - *Categorical branch*: `SimpleImputer(strategy="most_frequent")` fills NaN with the most common category, then `OneHotEncoder` creates binary indicator columns

3. The **outer `Pipeline`** chains these together: function transform → column transform → ready for any model.

`SimpleImputer` follows the same fit-on-train-only rule as `StandardScaler` - it learns the most frequent values from the training folds during `fit` and applies them during `transform`.

In [ ]:
# Update column lists after dropping duration and default
numeric_cols_final = [c for c in numeric_cols if c != "pdays"]
# pdays is handled separately by FunctionTransformer

# Categorical branch: impute missing → one-hot encode
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False)
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols_final),
        ("cat", cat_pipeline, categorical_cols),
    ],
    remainder="drop"
)

# Full preprocessing: pdays → binary flag, then ColumnTransformer
preprocess = make_pipeline(
    FunctionTransformer(pdays_to_contacted, validate=False),
    preprocessor
)

In [ ]:
# Verify imputation: what value fills the missing education?
preprocess.fit(X_train, y_train)
most_freq_edu = X_train["education"].mode()[0]
print(f"Row {sample_idx}: education was 'unknown' (now NaN), "
      f"imputed with most frequent value: '{most_freq_edu}'")

### Step 3: Model Selection via Cross-Validation

We compare models using cross-validation on the training data. Inside each fold, part of the training data is used for fitting and part for scoring. The training data plays both roles - this is normal and expected.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#### Baseline

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")

dummy_acc = cross_val_score(dummy, X_train, y_train, cv=cv, scoring="accuracy")
dummy_auc = cross_val_score(dummy, X_train, y_train, cv=cv, scoring="roc_auc")

print(f"Dummy (predict majority class):")
print(f"  Accuracy:  {dummy_acc.mean():.3f} ± {dummy_acc.std():.3f}")
print(f"  ROC AUC:   {dummy_auc.mean():.3f} ± {dummy_auc.std():.3f}")
print(f"\nAlways predicting 'no subscription' gets {1 - y_train.mean():.1%} accuracy.")
print(f"This matches 100% minus the {y_train.mean():.1%} positive class rate.")

Two baselines, two very different numbers. Accuracy looks high (88.7%) because the classes are imbalanced - always guessing "no" is right most of the time. ROC AUC is exactly 0.5 because a constant predictor has no ability to *rank* positive cases above negative ones. The ROC curve for a model with no discrimination is the diagonal line, and the area under it is 0.5. This is the true "no skill" baseline - any model that ranks better than a coin flip will exceed it. We use ROC AUC for the model comparison below because it measures ranking quality, not just overall correctness.

#### Compare Model Families

In [ ]:
import time

candidates = {
    "LogReg": make_pipeline(preprocess, LogisticRegression(max_iter=1000)),
    "KNN": make_pipeline(preprocess, KNeighborsClassifier()),
    "SVM (linear)": make_pipeline(preprocess, SVC(kernel="linear")),
    "SVM (rbf)": make_pipeline(preprocess, SVC(kernel="rbf")),
}

results = {}
for name, pipe in candidates.items():
    start = time.time()
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    elapsed = time.time() - start
    results[name] = {
        "mean_auc": scores.mean(),
        "std_auc": scores.std(),
        "time_s": elapsed
    }
    print(f"{name:15s}  AUC: {scores.mean():.3f} ± {scores.std():.3f}  ({elapsed:.1f}s)")

In [ ]:
results_df = pd.DataFrame(results).T.sort_values("mean_auc", ascending=False)
results_df

The outcome of this step is a **best model** - the model family and configuration that performed best in cross-validation. We're not keeping any of the fitted models from CV; those were trained on partial data (k-1 folds). We'll refit on all training data next.

##### Is it surprising that logistic regression wins? When would you expect SVM to do better?

Not necessarily. This is No Free Lunch in practice: model sophistication doesn't guarantee better performance on every dataset. After one-hot encoding, the feature space is high-dimensional and the relationship between features and target is approximately linear - exactly the regime where LogReg excels. SVM's advantage shows when the decision boundary is genuinely non-linear in the original feature space (as we saw with the moons dataset). On tabular data like this, SVM(rbf) is slower and no better than LogReg. Before looking at results, think about *why* a given model might win on *this* data - not which model is "better" in general.

### Step 4: Refit on All Training Data

Cross-validation told us *what* to build. Now we build it using all available training data, giving the model the maximum amount of data to learn from.

Notice how the pipeline makes this trivial: a single `.fit()` call handles all preprocessing and model fitting, with no risk of leakage or forgotten steps.

In [ ]:
# Use the best-performing model family from Step 3
# (adjust this based on actual CV results above)
best_pipeline = make_pipeline(
    preprocess,
    LogisticRegression(max_iter=1000)
)

best_pipeline.fit(X_train, y_train)
print("Best model refit on all training data.")
print(f"Training accuracy: {best_pipeline.score(X_train, y_train):.3f}")

### Step 5: Final Evaluation on Held-Out Test

This is the first and only time we use the test data. The score here is our estimate of how the model will perform on new, unseen data.

In [ ]:
y_pred = best_pipeline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["no", "yes"]))

##### How would you interpret this report to a business stakeholder?

The model correctly identifies 90% of cases overall, but it's much better at predicting non-subscribers (98% recall) than subscribers (22% recall). Out of every 100 actual subscribers, the model catches only 22. When it *does* predict "yes," it's right 58% of the time (precision). For a marketing campaign, this means the model misses most potential subscribers - we're being too conservative with our predictions.

#### Diagnostic Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0], cmap="Blues")
axes[0].set_title("Confusion Matrix")

# ROC curve
RocCurveDisplay.from_estimator(best_pipeline, X_test, y_test, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[1].set_title("ROC Curve")

plt.tight_layout()
plt.show()

##### Is an AUC of ~0.77 good?

It depends on the problem. AUC of 0.77 means that a randomly chosen subscriber is ranked higher than a randomly chosen non-subscriber 77% of the time. For a marketing campaign where the cost of a phone call is low and the value of a new subscriber is high, this might be sufficient - especially if we tune the threshold to catch more subscribers. For medical diagnosis, 0.77 would likely be insufficient. Context determines the standard.

### Threshold Tuning

The default 0.5 threshold treats false positives and false negatives as equally costly. In a marketing campaign, the costs are different: a wasted phone call (FP) is cheap; missing a potential subscriber (FN) means lost revenue.

We can tune the threshold using cross-validation on the training data - the test set stays untouched.

In [ ]:
tuned = TunedThresholdClassifierCV(
    make_pipeline(preprocess, LogisticRegression(max_iter=1000)),
    scoring="f1",
    cv=cv,
)
tuned.fit(X_train, y_train)
print(f"Optimal threshold: {tuned.best_threshold_:.3f}")

In [ ]:
y_pred_tuned = tuned.predict(X_test)

print("=== Default Threshold (0.5) ===")
print(classification_report(y_test, y_pred, target_names=["no", "yes"]))

print("\n=== Tuned Threshold ===")
print(classification_report(y_test, y_pred_tuned, target_names=["no", "yes"]))

##### The tuned model has lower accuracy. Is it actually better?

Yes, for this problem. The tuned model catches more subscribers (higher recall) at the cost of more false positives (lower precision on "no"). Accuracy drops because we're now incorrectly flagging some non-subscribers, but each caught subscriber is worth more than the cost of a wasted phone call. The right metric depends on the business context, not the overall accuracy number.

##### Why not just maximize recall directly?

A threshold near zero gives perfect recall by predicting "yes" for everyone, but precision collapses - you'd be calling every single person. F1 forces a balance between precision and recall, finding a threshold that actually improves the model rather than trivially gaming one metric.

### Calibration Check

Everything we did with threshold tuning depends on `predict_proba` returning meaningful probabilities. A well-calibrated model's predicted probabilities match observed frequencies - when it says 30%, roughly 30% of those cases are actually positive.

How to read a calibration plot: the diagonal line represents perfect calibration. Points above the diagonal mean the model *underestimates* the probability (predicted 20% but 40% were actually positive). Points below mean overestimates. What to look for:

- *Systematic S-curve*: the model is overconfident (probabilities too extreme) or underconfident (bunched toward the middle). This is the most common and most fixable pattern.
- *Consistent drift*: all points sit above or below the diagonal - the model consistently over- or under-predicts.
- *Single-bin noise*: individual bins that deviate while the overall trend follows the diagonal. This is expected with small test sets and is usually not a problem.

Like judging QQ plots for normality, calibration assessment is somewhat subjective. The question isn't "is this perfectly calibrated?" but "is this *systematically* miscalibrated in a way that would distort threshold decisions?"

In [ ]:
# Refit models for calibration comparison
logreg_pipe = make_pipeline(preprocess, LogisticRegression(max_iter=1000))
svm_pipe = make_pipeline(preprocess, SVC(kernel="rbf", probability=True))

logreg_pipe.fit(X_train, y_train)
svm_pipe.fit(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
CalibrationDisplay.from_estimator(logreg_pipe, X_test, y_test, ax=axes[0], n_bins=10)
axes[0].set_title("LogReg Calibration")
CalibrationDisplay.from_estimator(svm_pipe, X_test, y_test, ax=axes[1], n_bins=10)
axes[1].set_title("SVM (rbf) Calibration — Platt Scaling")
plt.tight_layout()
plt.show()

Logistic regression is typically well-calibrated because it directly optimizes log-likelihood - probabilities are a natural output of the model. SVM probabilities are estimated via Platt scaling (fitting a sigmoid to the decision function after training), which is an approximation and tends to be less reliable. Notice how the SVM curve deviates more from the diagonal.

When SVM calibration is poor, we can fix it with `CalibratedClassifierCV`, which uses cross-validation to learn a better mapping from decision function scores to probabilities.

In [ ]:
# Calibrate the SVM
svm_calibrated = CalibratedClassifierCV(
    SVC(kernel="rbf"), method="sigmoid", cv=5
)
svm_cal_pipe = make_pipeline(preprocess, svm_calibrated)
svm_cal_pipe.fit(X_train, y_train)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
CalibrationDisplay.from_estimator(logreg_pipe, X_test, y_test, ax=axes[0], n_bins=10)
axes[0].set_title("LogReg (no calibration needed)")
CalibrationDisplay.from_estimator(svm_pipe, X_test, y_test, ax=axes[1], n_bins=10)
axes[1].set_title("SVM — Platt Scaling (built-in)")
CalibrationDisplay.from_estimator(svm_cal_pipe, X_test, y_test, ax=axes[2], n_bins=10)
axes[2].set_title("SVM — CalibratedClassifierCV")
plt.tight_layout()
plt.show()

Compare the second and third panels: `CalibratedClassifierCV` should bring the SVM calibration curve closer to the diagonal. If you need to trust the probability outputs - for cost-based threshold tuning, for instance - calibration is worth the extra step. For LogReg on tabular data, it's usually unnecessary.

### Cost-Based Threshold Tuning

With calibrated probabilities, we can go beyond F1 and optimize directly for business costs. Suppose each phone call costs `$5` (FP cost) and each missed subscriber represents `$50` in lost revenue (FN cost). We can build a custom scorer that optimizes for total cost.

In [ ]:
def net_cost(y_true, y_pred, fp_cost=5, fn_cost=50):
    """Total cost of classification errors. Lower is better."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return fp * fp_cost + fn * fn_cost

# make_scorer expects higher = better, so negate the cost
cost_scorer = make_scorer(net_cost, greater_is_better=False)

In [ ]:
tuned_cost = TunedThresholdClassifierCV(
    make_pipeline(preprocess, LogisticRegression(max_iter=1000)),
    scoring=cost_scorer,
    cv=cv,
)
tuned_cost.fit(X_train, y_train)
print(f"Cost-optimal threshold: {tuned_cost.best_threshold_:.3f}")

In [ ]:
y_pred_cost = tuned_cost.predict(X_test)

print("=== Default (0.5) ===")
print(classification_report(y_test, y_pred, target_names=["no", "yes"]))

print("\n=== F1-Tuned ===")
print(classification_report(y_test, y_pred_tuned, target_names=["no", "yes"]))

print("\n=== Cost-Tuned ===")
print(classification_report(y_test, y_pred_cost, target_names=["no", "yes"]))

In [ ]:
# Compare total costs
for name, preds in [("Default (0.5)", y_pred), ("F1-Tuned", y_pred_tuned), ("Cost-Tuned", y_pred_cost)]:
    cost = net_cost(y_test, preds)
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    print(f"{name:18s}  FP: {fp:3d} x $5 = ${fp*5:,}   FN: {fn:3d} x $50 = ${fn*50:,}   Total: ${cost:,}")

##### The cost-tuned model has the lowest accuracy of the three. Why is it the best choice?

Because accuracy treats all errors as equal, but they're not. A missed subscriber costs 10x more than a wasted call. The cost-tuned model makes more FP errors (wasted calls at &#36;5 each) but catches far more subscribers (fewer FN errors at &#36;50 each). The total dollar cost is lower. This is exactly the principle from the Elkan paper (HW3b): separate estimation from decision-making, and let the business costs drive the threshold.

### Scaling Up: RandomizedSearchCV

In Step 3 we compared model families with default hyperparameters. In practice, you'd also tune hyperparameters - C for SVM, k for KNN, regularization strength for LogReg. `GridSearchCV` (from HW3) exhaustively searches every combination, but that gets expensive when the grid is large or the model is slow.

`RandomizedSearchCV` uses the same API but samples a fixed number of random combinations from the parameter space. You control the time budget with `n_iter` - regardless of how large the search space is, it runs exactly `n_iter` × `n_folds` fits.

This also demonstrates a key benefit of pipelines: the same `preprocess` pipeline works unchanged with any model or search strategy.

We also use `n_jobs=-1` here for the first time. This tells sklearn to use all available CPU cores in parallel - each fold/parameter combination runs on a separate core. For a search with 75 fits, this can cut wall time significantly.

In [ ]:
from scipy.stats import loguniform

# Search over SVM hyperparameters using the same preprocessing pipeline
svm_pipe = make_pipeline(preprocess, SVC())

param_distributions = {
    "svc__C": loguniform(0.01, 100),       # log-uniform: samples across orders of magnitude
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": ["scale", "auto"],        # only relevant for rbf
}

search = RandomizedSearchCV(
    svm_pipe,
    param_distributions,
    n_iter=15,
    cv=5,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
)

start = time.time()
search.fit(X_train, y_train)
elapsed = time.time() - start

print(f"RandomizedSearchCV: 15 combinations x 5 folds = 60 fits in {elapsed:.1f}s")
print(f"Best AUC: {search.best_score_:.3f}")
print(f"Best params: {search.best_params_}")

Compare this to our earlier SVM results with default hyperparameters. The pipeline handled all preprocessing identically - we only changed the search strategy wrapping the model.

What happens if you increase `n_iter`? Try values of 30 and 50. You'll see AUC gains tail off while runtime jumps dramatically - 3x and 200x longer. This isn't just from running more fits. More iterations sample more of the `loguniform` distribution, hitting larger C values. Large C forces a tighter margin with more support vectors, and SVM's $O(n^2)$ training cost makes each of those fits much slower. The iteration count controls not just how many fits you run, but *where* in the parameter space you explore.

---

## Workflow Summary

This is the workflow for your project. What changes: the dataset, the models (you'll add trees and ensembles), and the business question. The structure stays the same.

1. **Hold out test data** - `train_test_split(stratify=y)`
2. **Explore and engineer** - EDA, drop leaky features, build `ColumnTransformer` (training data only)
3. **Select model via cross-validation** - compare families, tune hyperparameters; outcome: best model (family + hyperparams)
4. **Refit on all training data** - outcome: best fit (family + hyperparams + learned params)
5. **Final evaluation on test** - one shot; this is your real-world performance estimate